In [2]:
# ============================================================
#   FILTER PEROVSKITE ABX3 - Lokal Laptop
#   Jalankan script ini langsung dengan: python filter_perovskite_local.py
# ============================================================

import pandas as pd
import numpy as np
import re
import os
import shutil


# ── KONFIGURASI ───────────────────────────────────────────

# *** SESUAIKAN SEMUA PATH DI BAWAH INI SESUAI LOKASI DI LAPTOP KAMU ***

# --- CSV ---
TRAIN_CSV    = r"c:/Users/HP/Downloads/dataset CPICANN/anno_train.csv"
VAL_CSV      = r"c:/Users/HP/Downloads/dataset CPICANN/anno_val.csv"

# --- Folder INPUT gambar (bisa beda-beda lokasi, tidak harus satu induk) ---
TRAIN_FOLDER = r"c:/Users/HP/Downloads/dataset CPICANN/train_with_back/train_with_back"
VAL_FOLDER   = r"c:/Users/HP/Downloads/dataset CPICANN/val_with_back/val_with_back"

# --- Folder OUTPUT hasil filter (akan dibuat otomatis jika belum ada) ---
TRAIN_OUT    = r"c:/Users/HP/Downloads/dataset CPICANN/train_with_back_perovskiteABX3"
VAL_OUT      = r"c:/Users/HP/Downloads/dataset CPICANN/val_with_back_perovskiteABX3"


# ── NAMA KOLOM ────────────────────────────────────────────

COL_FORMULA     = "formula"
COL_A           = "a"
COL_B           = "b"
COL_C           = "c"
COL_ALPHA       = "alpha"
COL_BETA        = "beta"
COL_GAMMA       = "gamma"
COL_CRYSTAL_SYS = "crystalSys"
COL_DATA_ID     = "dataId"

# Mapping crystalSys → nama sistem kristal
CRYSTAL_SYS_MAP = {
    0: "Cubic",
    1: "Tetragonal",
    2: "Orthorhombic",
    5: "Monoclinic",
}
TARGET_CRYSTAL_SYS = [0, 1, 2, 5]


# ── DEFINISI ELEMEN PER POSISI ────────────────────────────

A_SITE = {
    "Li", "Na", "K", "Rb", "Cs",
    "Mg", "Ca", "Sr", "Ba",
    "La", "Ce", "Pr", "Nd", "Sm", "Eu",
    "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu",
    "Y", "Bi", "Th", "Tl", "Ag", "Cd",
}

B_SITE = {
    "Sc", "Ti", "V",  "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Nb", "Mo", "Ru", "Rh", "Pd", "Ag",
    "Hf", "Ta", "W",  "Re", "Os", "Ir", "Pt",
    "Al", "Ga", "In", "Ge", "Sn", "Pb", "Bi",
    "Zr", "Si", "Mg", "Cd", "Sc",
}

X_SITE = {
    "O",
    "F", "Cl", "Br", "I",
    "S", "Se", "Te",
    "N", "P",
}


# ── FUNGSI FILTER ABX3 ────────────────────────────────────

def parse_formula(formula: str) -> dict:
    pattern = r"([A-Z][a-z]?)(\d*\.?\d*)"
    elements = {}
    for elem, count in re.findall(pattern, str(formula)):
        count = float(count) if count else 1.0
        elements[elem] = elements.get(elem, 0) + count
    return elements


def is_ABX3(formula: str) -> tuple:
    if not formula or str(formula).strip() in ("", "nan"):
        return False, "Formula kosong"
    elems = parse_formula(formula)
    if not elems:
        return False, "Tidak bisa di-parse"

    unknown = {e for e in elems if e not in A_SITE | B_SITE | X_SITE}
    if unknown:
        return False, f"Elemen tidak dikenali: {unknown}"

    a_elems = {e: n for e, n in elems.items()
               if e in A_SITE and e not in B_SITE}
    b_elems = {e: n for e, n in elems.items() if e in B_SITE}
    x_elems = {e: n for e, n in elems.items() if e in X_SITE}

    n_a = sum(a_elems.values())
    n_b = sum(b_elems.values())
    n_x = sum(x_elems.values())

    if n_a == 0:
        return False, "Tidak ada elemen A-site"
    if n_b == 0:
        return False, "Tidak ada elemen B-site"
    if n_x == 0:
        return False, "Tidak ada elemen X-site"

    ratio_a = n_a / n_b
    ratio_x = n_x / n_b

    if abs(ratio_a - 1.0) <= 0.15 and abs(ratio_x - 3.0) <= 0.30:
        return True, "ABX3"

    return False, f"Rasio A:B:X = {n_a:.2f}:{n_b:.2f}:{n_x:.2f} (bukan ABX3)"


# ── FUNGSI PROSES PER SPLIT ───────────────────────────────

def process_split(csv_path: str, src_folder: str, out_folder: str, split_name: str):
    print(f"\n{'='*60}")
    print(f"  PROSES SPLIT: {split_name.upper()}")
    print(f"{'='*60}")
    print(f"  CSV sumber    : {csv_path}")
    print(f"  Folder input  : {src_folder}")
    print(f"  Folder output : {out_folder}")

    # 1. Load CSV
    if not os.path.exists(csv_path):
        print(f"❌ File CSV tidak ditemukan: {csv_path}")
        return
    df = pd.read_csv(csv_path)
    print(f"\n✅ CSV dimuat  → {len(df)} baris, {len(df.columns)} kolom")
    print(f"   Kolom: {list(df.columns)}")

    # 2. Validasi kolom wajib
    required_cols = [COL_FORMULA, COL_CRYSTAL_SYS, COL_DATA_ID]
    missing_cols  = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        print(f"❌ Kolom wajib tidak ditemukan: {missing_cols}")
        return

    # 3. Filter ABX3 berdasarkan formula
    results = df[COL_FORMULA].apply(is_ABX3)
    df["is_ABX3"]   = results.apply(lambda x: x[0])
    df["abx3_note"] = results.apply(lambda x: x[1])
    df_perov = df[df["is_ABX3"]].copy()

    print(f"\n📊 Total data semula          : {len(df)}")
    print(f"💎 Setelah filter ABX3        : {len(df_perov)}")

    # 4. Filter sistem kristal
    df_perov["crystal_sys_label"] = df_perov[COL_CRYSTAL_SYS].map(CRYSTAL_SYS_MAP)
    df_filtered = df_perov[df_perov[COL_CRYSTAL_SYS].isin(TARGET_CRYSTAL_SYS)].copy()

    print(f"\n🔍 Setelah filter crystalSys {TARGET_CRYSTAL_SYS}:")
    print(f"   Jumlah baris : {len(df_filtered)}")

    dist = (
        df_filtered
        .groupby([COL_CRYSTAL_SYS, "crystal_sys_label"])
        .size()
        .reset_index(name="jumlah")
        .sort_values(COL_CRYSTAL_SYS)
    )
    print()
    print("─── Distribusi Sistem Kristal ──────────────────────")
    print(dist.to_string(index=False))

    # 5. Kumpulkan dataId hasil filter
    data_ids = df_filtered[COL_DATA_ID].astype(str).tolist()
    data_id_set = set(data_ids)
    print(f"\n🗂️  Total dataId hasil filter  : {len(data_id_set)}")

    # 6. Cek ketersediaan source folder
    if not os.path.isdir(src_folder):
        print(f"❌ Folder sumber tidak ditemukan: {src_folder}")
        return

    files_in_src = os.listdir(src_folder)
    folder_file_map = {}
    for fname in files_in_src:
        name_no_ext, _ = os.path.splitext(fname)
        folder_file_map[str(name_no_ext)] = fname

    print(f"📁 Total file di folder sumber : {len(files_in_src)}")

    # 7. Buat folder output
    os.makedirs(out_folder, exist_ok=True)
    print(f"\n📂 Output folder : {out_folder}")

    # 8. Salin file yang sesuai dataId ke folder output
    copied  = 0
    missing = []

    for data_id in data_id_set:
        if data_id in folder_file_map:
            src_file  = os.path.join(src_folder, folder_file_map[data_id])
            dest_file = os.path.join(out_folder, folder_file_map[data_id])
            if not os.path.exists(dest_file):           # skip jika sudah ada
                shutil.copy2(src_file, dest_file)
            copied += 1
        else:
            missing.append(data_id)

    # 9. Ringkasan hasil salin
    print(f"\n─── Hasil Penyalinan ───────────────────────────────")
    print(f"   ✅ File berhasil disalin  : {copied}")
    print(f"   ❌ dataId tidak ditemukan : {len(missing)}")
    print(f"   📁 Total file di output  : {len(os.listdir(out_folder))}")

    if missing:
        print(f"\n⚠️  Daftar dataId TIDAK ditemukan (maks 50 ditampilkan):")
        for mid in missing[:50]:
            print(f"   - {mid}")
        if len(missing) > 50:
            print(f"   ... dan {len(missing) - 50} lainnya")
    else:
        print(f"\n🎉 Semua file dataId berhasil disalin ke '{os.path.basename(out_folder)}'!")

    return df_filtered


# ── MAIN ──────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 60)
    print("  FILTER PEROVSKITE ABX3 — LOKAL LAPTOP")
    print("=" * 60)
    print(f"  TRAIN CSV     : {TRAIN_CSV}")
    print(f"  TRAIN INPUT   : {TRAIN_FOLDER}")
    print(f"  TRAIN OUTPUT  : {TRAIN_OUT}")
    print(f"  VAL CSV       : {VAL_CSV}")
    print(f"  VAL INPUT     : {VAL_FOLDER}")
    print(f"  VAL OUTPUT    : {VAL_OUT}")

    df_train = process_split(
        csv_path   = TRAIN_CSV,
        src_folder = TRAIN_FOLDER,
        out_folder = TRAIN_OUT,
        split_name = "train",
    )

    df_val = process_split(
        csv_path   = VAL_CSV,
        src_folder = VAL_FOLDER,
        out_folder = VAL_OUT,
        split_name = "val",
    )

    print(f"\n{'='*60}")
    print(f"  SELESAI")
    if df_train is not None:
        print(f"  train_with_back_allperovskite : {len(df_train)} entri")
    if df_val is not None:
        print(f"  val_with_back_allperovskite   : {len(df_val)} entri")
    print(f"{'='*60}")

  FILTER PEROVSKITE ABX3 — LOKAL LAPTOP
  TRAIN CSV     : c:/Users/HP/Downloads/dataset CPICANN/anno_train.csv
  TRAIN INPUT   : c:/Users/HP/Downloads/dataset CPICANN/train_with_back/train_with_back
  TRAIN OUTPUT  : c:/Users/HP/Downloads/dataset CPICANN/train_with_back_perovskiteABX3
  VAL CSV       : c:/Users/HP/Downloads/dataset CPICANN/anno_val.csv
  VAL INPUT     : c:/Users/HP/Downloads/dataset CPICANN/val_with_back/val_with_back
  VAL OUTPUT    : c:/Users/HP/Downloads/dataset CPICANN/val_with_back_perovskiteABX3

  PROSES SPLIT: TRAIN
  CSV sumber    : c:/Users/HP/Downloads/dataset CPICANN/anno_train.csv
  Folder input  : c:/Users/HP/Downloads/dataset CPICANN/train_with_back/train_with_back
  Folder output : c:/Users/HP/Downloads/dataset CPICANN/train_with_back_perovskiteABX3

✅ CSV dimuat  → 553752 baris, 13 kolom
   Kolom: ['dataId', 'No', 'formula', 'symbolSet', 'spaceGroup', 'spaceGroupNo', 'crystalSys', 'a', 'b', 'c', 'alpha', 'beta', 'gamma']

📊 Total data semula          :